In [2]:
import pandas as pd
import numpy as np
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

In [3]:
import sklearn.neighbors._base
import sys
sys.modules['sklearn.neighbors.base'] = sklearn.neighbors._base

In [4]:
from missingpy import MissForest
imputer = MissForest()

In [5]:
n_items_list = [50, 100, 100, 100, 500, 500, 1000, 1000, 1000, 1500, 1500]
m_users_list = [50, 100, 300, 500, 100, 500, 100, 200, 300, 1000, 2000]

for n_items, m_users in zip(n_items_list, m_users_list):
    # Your code using n_items and m_users
    # For example:
    print(f"n_items: {n_items}, m_users: {m_users}")
    # Run your analysis or processing using n_items and m_users here


n_items: 50, m_users: 50
n_items: 100, m_users: 100
n_items: 100, m_users: 300
n_items: 100, m_users: 500
n_items: 500, m_users: 100
n_items: 500, m_users: 500
n_items: 1000, m_users: 100
n_items: 1000, m_users: 200
n_items: 1000, m_users: 300
n_items: 1500, m_users: 1000
n_items: 1500, m_users: 2000


In [6]:
#main loop 
import time as time_lib

k = 10 # k-fold validation 
n_items_list = [ 100, 100, 100, 100]#, 500, 500, 500, 500, 500, 1000, 1000, 1000, 1000, 1000]
m_users_list = [ 100, 300, 500, 700]#, 50, 100, 300, 500, 700,  50, 100, 300, 500, 700 ]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])
# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        #print(lab)
        df = pd.read_csv(url_dict[lab])
        #print(lab)
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
    count = 0 

    for lab in labs_test:
        count += 1
        #print(lab)
        
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        #df, maxs = normal1(masked_df)
        #df, maxs = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df)

        from missingpy import MissForest
        imputer = MissForest(verbose=0)

        start = time_lib.time()

        imputed = None 

        # Set a time limit of 3600 seconds (1 hour)
        time_limit = 30#3600
        
        try: 
            imputed = pd.DataFrame(imputer.fit_transform(df))
        except TimeoutError:
            print("at (n,m)= ("+str(n)+" ,"+str(m)+")")
            print("Imputation time exceeded the limit of {} seconds.".format(time_limit))
            imputed = None

        if imputed is not None:
            # Rounding for when no normalization           
            for i in range(mm):
                for j in range(nn):
                    x = imputed.iloc[i,j]
                    if x < 1:
                        imputed.iloc[i,j] = 1
                    elif x > col_max[j]:
                        imputed.iloc[i,j] = col_max[j]
                    else:
                        imputed.iloc[i,j] = round(x,0)

            #imputed = unnormal1(imputed, maxs)
            #imputed = unnormal2(imputed, maxs)
            #imputed = unnormal3(imputed, maxs, probs)

            end = time_lib.time()
            time = end - start

            df_orig.index = range(mm)
            imputed.index = range(mm)

            df_orig.columns = range(nn)
            imputed.columns = range(nn)

            #save the result data
            url = './result/'

            if count<10:
                filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_soft_imputed.csv'
            else:
                filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_soft_imputed.csv'
            
            imputed.to_csv(filename)
            print("saved the result file as "+str(filename))
            diff = df_orig - imputed
            sq_diff = diff ** 2
            abs_diff = abs(diff)

            n_match = 0
            for i in range(mm):
                for j in range(nn):
                    if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                        n_match += 1
            acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
            rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
            mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
            
            acc_list.append(acc)
            rmse_list.append(rmse)
            mad_list.append(mad)
            time_list.append(time)
        else: #if time exceeds the limit 
            acc = 0 
            rmse = 0 
            mad = 0 
            time = time_limit    

            acc_list.append(acc)
            rmse_list.append(rmse)
            mad_list.append(mad)
            time_list.append(time)


    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a) 

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)

    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_missforest_imputed.csv'
    df_temp.to_csv(filename)
    print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'missforest_imputed.csv'
    df_temp.to_csv(filename)
    print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)
        

        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/100-by-100_original_matrix.csv
./data/20230808_100-by-100_10_fold_test_data01.csv
./data/20230808_100-by-100_10_fold_test_data02.csv
./data/20230808_100-by-100_10_fold_test_data03.csv
./data/20230808_100-by-100_10_fold_test_data04.csv
./data/20230808_100-by-100_10_fold_test_data05.csv
./data/20230808_100-by-100_10_fold_test_data06.csv
./data/20230808_100-by-100_10_fold_test_data07.csv
./data/20230808_100-by-100_10_fold_test_data08.csv
./data/20230808_100-by-100_10_fold_test_data09.csv
./data/20230808_100-by-100_10_fold_test_data10.csv
sparsity: 0.2914
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
saved the result file as ./result/20230808_100-by-100_10_fold_test_data01_soft_imputed.csv
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7
Iteration: 8
saved the result file as ./

'n: 100 ,m: 100'

,0,1,2,3
0,0.449153,0.925929,0.645480,87.676472
1,0.473164,0.938806,0.635593,109.491697
2,0.433616,0.966238,0.676554,70.569593
3,0.446328,0.946298,0.655367,59.680877
4,0.445698,0.942643,0.657264,94.827224
5,0.464034,0.941894,0.647391,106.204058
6,0.486601,0.911454,0.607898,59.205433
7,0.449929,0.953059,0.660085,94.257085
8,0.452750,0.950095,0.648801,107.489527
9,0.468265,0.909129,0.623413,58.980614


saved 20230808_100-by-100the summary result file as./result/20230808_100-by-100_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_100-by-100_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,100.0,0.2914,0.456954,0.015748,0.938554,0.018128,0.645785,0.019512,84.838258,20.912693


...

,0,1,2,3
0,0.440380,0.979863,0.679956,717.934159
1,0.457206,0.988226,0.673738,809.858986
2,0.450256,0.983403,0.675201,897.237206
3,0.451554,0.976319,0.670932,625.295693
4,0.458135,0.964829,0.659598,482.776901
5,0.443510,0.972567,0.675320,562.542699
6,0.443510,0.992476,0.685923,804.350493
7,0.444973,0.966533,0.671664,805.353799
8,0.446435,0.991186,0.684095,484.224471
9,0.442779,0.995786,0.688848,724.669883


saved 20230808_100-by-700the summary result file as./result/20230808_100-by-700_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_100-by-700_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,100.0,0.291400,0.456954,0.015748,0.938554,0.018128,0.645785,0.019512,84.838258,20.912693
1,100.0,300.0,0.420200,0.448086,0.007135,0.964261,0.011537,0.667184,0.010903,274.891377,59.331898
2,100.0,500.0,0.526960,0.440385,0.011870,0.989405,0.018748,0.686665,0.014093,500.320981,36.462388
3,100.0,700.0,0.609329,0.447874,0.006167,0.981119,0.010939,0.676528,0.008574,691.424429,145.821424


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,100.0,0.291400,0.456954,0.015748,0.938554,0.018128,0.645785,0.019512,84.838258,20.912693
1,100.0,300.0,0.420200,0.448086,0.007135,0.964261,0.011537,0.667184,0.010903,274.891377,59.331898
2,100.0,500.0,0.526960,0.440385,0.011870,0.989405,0.018748,0.686665,0.014093,500.320981,36.462388
3,100.0,700.0,0.609329,0.447874,0.006167,0.981119,0.010939,0.676528,0.008574,691.424429,145.821424


,n_items,m_users,acc,rmse,mad,time
0,100,100,0.449153,0.925929,0.645480,87.676472
1,100,100,0.473164,0.938806,0.635593,109.491697
2,100,100,0.433616,0.966238,0.676554,70.569593
3,100,100,0.446328,0.946298,0.655367,59.680877
4,100,100,0.445698,0.942643,0.657264,94.827224
5,100,100,0.464034,0.941894,0.647391,106.204058
6,100,100,0.486601,0.911454,0.607898,59.205433
7,100,100,0.449929,0.953059,0.660085,94.257085
8,100,100,0.452750,0.950095,0.648801,107.489527
9,100,100,0.468265,0.909129,0.623413,58.980614


In [7]:
#main loop 
import time as time_lib

k = 10 # k-fold validation 
n_items_list = [ 500,500, 500, 500, 500]#, 500, 500, 500, 500, 500, 1000, 1000, 1000, 1000, 1000]
m_users_list = [ 50, 100, 300, 500, 700]#, 50, 100, 300, 500, 700,  50, 100, 300, 500, 700 ]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])
# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        #print(lab)
        df = pd.read_csv(url_dict[lab])
        #print(lab)
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
    count = 0 

    for lab in labs_test:
        count += 1
        #print(lab)
        
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        #df, maxs = normal1(masked_df)
        #df, maxs = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df)

        from missingpy import MissForest
        imputer = MissForest(verbose=0)

        start = time_lib.time()

        imputed = None 

        # Set a time limit of 3600 seconds (1 hour)
        time_limit = 30#3600
        
        try: 
            imputed = pd.DataFrame(imputer.fit_transform(df))
        except TimeoutError:
            print("at (n,m)= ("+str(n)+" ,"+str(m)+")")
            print("Imputation time exceeded the limit of {} seconds.".format(time_limit))
            imputed = None

        if imputed is not None:
            # Rounding for when no normalization           
            for i in range(mm):
                for j in range(nn):
                    x = imputed.iloc[i,j]
                    if x < 1:
                        imputed.iloc[i,j] = 1
                    elif x > col_max[j]:
                        imputed.iloc[i,j] = col_max[j]
                    else:
                        imputed.iloc[i,j] = round(x,0)

            #imputed = unnormal1(imputed, maxs)
            #imputed = unnormal2(imputed, maxs)
            #imputed = unnormal3(imputed, maxs, probs)

            end = time_lib.time()
            time = end - start

            df_orig.index = range(mm)
            imputed.index = range(mm)

            df_orig.columns = range(nn)
            imputed.columns = range(nn)

            #save the result data
            url = './result/'

            if count<10:
                filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_soft_imputed.csv'
            else:
                filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_soft_imputed.csv'
            
            imputed.to_csv(filename)
            print("saved the result file as "+str(filename))
            diff = df_orig - imputed
            sq_diff = diff ** 2
            abs_diff = abs(diff)

            n_match = 0
            for i in range(mm):
                for j in range(nn):
                    if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                        n_match += 1
            acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
            rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
            mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
            
            acc_list.append(acc)
            rmse_list.append(rmse)
            mad_list.append(mad)
            time_list.append(time)
        else: #if time exceeds the limit 
            acc = 0 
            rmse = 0 
            mad = 0 
            time = time_limit    

            acc_list.append(acc)
            rmse_list.append(rmse)
            mad_list.append(mad)
            time_list.append(time)


    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a) 

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)

    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_missforest_imputed.csv'
    df_temp.to_csv(filename)
    print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'missforest_imputed.csv'
    df_temp.to_csv(filename)
    print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)
        

        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/500-by-50_original_matrix.csv
./data/20230808_500-by-50_10_fold_test_data01.csv
./data/20230808_500-by-50_10_fold_test_data02.csv
./data/20230808_500-by-50_10_fold_test_data03.csv
./data/20230808_500-by-50_10_fold_test_data04.csv
./data/20230808_500-by-50_10_fold_test_data05.csv
./data/20230808_500-by-50_10_fold_test_data06.csv
./data/20230808_500-by-50_10_fold_test_data07.csv
./data/20230808_500-by-50_10_fold_test_data08.csv
./data/20230808_500-by-50_10_fold_test_data09.csv
./data/20230808_500-by-50_10_fold_test_data10.csv
sparsity: 0.46664000000000005
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7
Iteration: 8
Iteration: 9
saved the result file as ./result/20230808_500-by-50_10_fold_test_data01_soft_imputed.csv
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7


'n: 500 ,m: 50'

,0,1,2,3
0,0.411103,1.039721,0.732933,69.718097
1,0.442611,0.993980,0.684921,73.073320
2,0.425356,1.030299,0.714929,71.805848
3,0.432108,0.955106,0.672168,50.561885
4,0.432108,1.023358,0.714179,50.273343
5,0.423106,0.997747,0.704426,71.797614
6,0.426537,1.010440,0.701649,56.334573
7,0.429535,0.989828,0.694903,71.569408
8,0.411544,1.058272,0.745127,69.861958
9,0.419040,1.024805,0.723388,73.564869


saved 20230808_500-by-50the summary result file as./result/20230808_500-by-50_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_500-by-50_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,500.0,50.0,0.46664,0.425305,0.009699,1.012356,0.02943,0.708862,0.022002,65.856092,9.50563


...

,0,1,2,3
0,0.416380,1.074961,0.751411,1237.970306
1,0.418858,1.070212,0.747557,1325.927671
2,0.424501,1.057077,0.736958,1255.101901
3,0.420727,1.069753,0.744426,1253.515574
4,0.423479,1.052764,0.733691,1257.146712
5,0.416460,1.074182,0.751445,1253.939450
6,0.418112,1.066467,0.747592,1257.751095
7,0.418662,1.071103,0.747592,1254.643031
8,0.421828,1.056614,0.737682,1249.763447
9,0.420039,1.054984,0.737820,1259.800726


saved 20230808_500-by-700the summary result file as./result/20230808_500-by-700_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_500-by-700_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,500.0,50.0,0.466640,0.425305,0.009699,1.012356,0.029430,0.708862,0.022002,65.856092,9.505630
1,500.0,100.0,0.530200,0.434525,0.012611,0.992451,0.024124,0.691997,0.021442,121.655184,20.892622
2,500.0,300.0,0.661573,0.424356,0.008619,1.033550,0.005953,0.722421,0.008051,479.364078,33.584007
3,500.0,500.0,0.740172,0.422803,0.006388,1.062164,0.009222,0.739705,0.007992,885.274006,65.043270
4,500.0,700.0,0.792409,0.419904,0.002758,1.064812,0.008534,0.743617,0.006511,1260.555991,23.753751


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,500.0,50.0,0.466640,0.425305,0.009699,1.012356,0.029430,0.708862,0.022002,65.856092,9.505630
1,500.0,100.0,0.530200,0.434525,0.012611,0.992451,0.024124,0.691997,0.021442,121.655184,20.892622
2,500.0,300.0,0.661573,0.424356,0.008619,1.033550,0.005953,0.722421,0.008051,479.364078,33.584007
3,500.0,500.0,0.740172,0.422803,0.006388,1.062164,0.009222,0.739705,0.007992,885.274006,65.043270
4,500.0,700.0,0.792409,0.419904,0.002758,1.064812,0.008534,0.743617,0.006511,1260.555991,23.753751


,n_items,m_users,acc,rmse,mad,time
0,500,50,0.411103,1.039721,0.732933,69.718097
1,500,50,0.442611,0.993980,0.684921,73.073320
2,500,50,0.425356,1.030299,0.714929,71.805848
3,500,50,0.432108,0.955106,0.672168,50.561885
4,500,50,0.432108,1.023358,0.714179,50.273343
5,500,50,0.423106,0.997747,0.704426,71.797614
6,500,50,0.426537,1.010440,0.701649,56.334573
7,500,50,0.429535,0.989828,0.694903,71.569408
8,500,50,0.411544,1.058272,0.745127,69.861958
9,500,50,0.419040,1.024805,0.723388,73.564869


In [8]:
#main loop 
import time as time_lib

k = 10 # k-fold validation 
n_items_list = [ 1000,1000, 1000, 1000, 1000]#, 500, 500, 500, 500, 500, 1000, 1000, 1000, 1000, 1000]
m_users_list = [ 50, 100, 300, 500, 700]#, 50, 100, 300, 500, 700,  50, 100, 300, 500, 700 ]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])
# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        #print(lab)
        df = pd.read_csv(url_dict[lab])
        #print(lab)
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
    count = 0 

    for lab in labs_test:
        count += 1
        #print(lab)
        
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        #df, maxs = normal1(masked_df)
        #df, maxs = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df)

        from missingpy import MissForest
        imputer = MissForest(verbose=0)

        start = time_lib.time()

        imputed = None 

        # Set a time limit of 3600 seconds (1 hour)
        time_limit = 30#3600
        
        try: 
            imputed = pd.DataFrame(imputer.fit_transform(df))
        except TimeoutError:
            print("at (n,m)= ("+str(n)+" ,"+str(m)+")")
            print("Imputation time exceeded the limit of {} seconds.".format(time_limit))
            imputed = None

        if imputed is not None:
            # Rounding for when no normalization           
            for i in range(mm):
                for j in range(nn):
                    x = imputed.iloc[i,j]
                    if x < 1:
                        imputed.iloc[i,j] = 1
                    elif x > col_max[j]:
                        imputed.iloc[i,j] = col_max[j]
                    else:
                        imputed.iloc[i,j] = round(x,0)

            #imputed = unnormal1(imputed, maxs)
            #imputed = unnormal2(imputed, maxs)
            #imputed = unnormal3(imputed, maxs, probs)

            end = time_lib.time()
            time = end - start

            df_orig.index = range(mm)
            imputed.index = range(mm)

            df_orig.columns = range(nn)
            imputed.columns = range(nn)

            #save the result data
            url = './result/'

            if count<10:
                filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_soft_imputed.csv'
            else:
                filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_soft_imputed.csv'
            
            imputed.to_csv(filename)
            print("saved the result file as "+str(filename))
            diff = df_orig - imputed
            sq_diff = diff ** 2
            abs_diff = abs(diff)

            n_match = 0
            for i in range(mm):
                for j in range(nn):
                    if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                        n_match += 1
            acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
            rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
            mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
            
            acc_list.append(acc)
            rmse_list.append(rmse)
            mad_list.append(mad)
            time_list.append(time)
        else: #if time exceeds the limit 
            acc = 0 
            rmse = 0 
            mad = 0 
            time = time_limit    

            acc_list.append(acc)
            rmse_list.append(rmse)
            mad_list.append(mad)
            time_list.append(time)


    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a) 

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)

    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_missforest_imputed.csv'
    df_temp.to_csv(filename)
    print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'missforest_imputed.csv'
    df_temp.to_csv(filename)
    print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)
        

        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/1000-by-50_original_matrix.csv
./data/20230808_1000-by-50_10_fold_test_data01.csv
./data/20230808_1000-by-50_10_fold_test_data02.csv
./data/20230808_1000-by-50_10_fold_test_data03.csv
./data/20230808_1000-by-50_10_fold_test_data04.csv
./data/20230808_1000-by-50_10_fold_test_data05.csv
./data/20230808_1000-by-50_10_fold_test_data06.csv
./data/20230808_1000-by-50_10_fold_test_data07.csv
./data/20230808_1000-by-50_10_fold_test_data08.csv
./data/20230808_1000-by-50_10_fold_test_data09.csv
./data/20230808_1000-by-50_10_fold_test_data10.csv
sparsity: 0.63338
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7
Iteration: 8
saved the result file as ./result/20230808_1000-by-50_10_fold_test_data01_soft_imputed.csv
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5
Iteration: 6
Iteration: 7
Iteration: 8


'n: 1000 ,m: 50'

,0,1,2,3
0,0.420076,1.036697,0.724495,73.491650
1,0.432079,1.041423,0.720131,84.029366
2,0.405346,1.110875,0.779051,67.089793
3,0.400982,1.043777,0.743590,83.205128
4,0.422259,1.046909,0.734861,84.748513
5,0.408620,1.060371,0.752319,44.604665
6,0.440807,1.030363,0.704855,58.642327
7,0.418985,1.069337,0.747409,71.868041
8,0.437534,1.036434,0.713039,72.309907
9,0.401854,1.062394,0.756816,72.074476


saved 20230808_1000-by-50the summary result file as./result/20230808_1000-by-50_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_1000-by-50_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,1000.0,50.0,0.63338,0.418854,0.014585,1.053858,0.023712,0.737656,0.022598,71.206387,12.391183


...

,0,1,2,3
0,0.411588,1.093731,0.766456,1529.510779
1,0.411588,1.104889,0.773449,1530.028299
2,0.411477,1.085480,0.761128,1526.978428
3,0.421800,1.072104,0.746698,1521.836091
4,0.411921,1.080046,0.757576,1527.760422
5,0.417027,1.098289,0.762460,1517.889727
6,0.414918,1.096316,0.765235,1532.618173
7,0.417092,1.104476,0.768590,1513.787917
8,0.410211,1.091487,0.766038,1523.949278
9,0.421865,1.085317,0.753052,1641.048975


saved 20230808_1000-by-700the summary result file as./result/20230808_1000-by-700_10_fold_test_data_all_missforest_imputed.csv
saved the summary result file as ./result/20230808_1000-by-700_missforest_imputed.csv


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,1000.0,50.0,0.633380,0.418854,0.014585,1.053858,0.023712,0.737656,0.022598,71.206387,12.391183
1,1000.0,100.0,0.686710,0.428229,0.008963,1.036829,0.013175,0.720930,0.013065,150.444715,17.793127
2,1000.0,300.0,0.785123,0.419032,0.008676,1.072662,0.014723,0.748646,0.014071,591.091679,24.052808
3,1000.0,500.0,0.837624,0.415517,0.004787,1.089762,0.010194,0.761258,0.008374,1043.780468,44.529281
4,1000.0,700.0,0.871296,0.414949,0.004345,1.091213,0.010556,0.762068,0.007859,1536.540809,37.175327


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,1000.0,50.0,0.633380,0.418854,0.014585,1.053858,0.023712,0.737656,0.022598,71.206387,12.391183
1,1000.0,100.0,0.686710,0.428229,0.008963,1.036829,0.013175,0.720930,0.013065,150.444715,17.793127
2,1000.0,300.0,0.785123,0.419032,0.008676,1.072662,0.014723,0.748646,0.014071,591.091679,24.052808
3,1000.0,500.0,0.837624,0.415517,0.004787,1.089762,0.010194,0.761258,0.008374,1043.780468,44.529281
4,1000.0,700.0,0.871296,0.414949,0.004345,1.091213,0.010556,0.762068,0.007859,1536.540809,37.175327


,n_items,m_users,acc,rmse,mad,time
0,1000,50,0.420076,1.036697,0.724495,73.491650
1,1000,50,0.432079,1.041423,0.720131,84.029366
2,1000,50,0.405346,1.110875,0.779051,67.089793
3,1000,50,0.400982,1.043777,0.743590,83.205128
4,1000,50,0.422259,1.046909,0.734861,84.748513
5,1000,50,0.408620,1.060371,0.752319,44.604665
6,1000,50,0.440807,1.030363,0.704855,58.642327
7,1000,50,0.418985,1.069337,0.747409,71.868041
8,1000,50,0.437534,1.036434,0.713039,72.309907
9,1000,50,0.401854,1.062394,0.756816,72.074476
